# Validation Test Execution Notebook
### Air Quality Review (AQR) System â€” GAMP 5 Category 5 Validation Protocol

This notebook contains individual cells to execute and verify every test case in the combined validation protocol (Appendix 2 and 3).
All test data is prepared by copying and programmatically editing real files from the project's `data/` directory to ensure high-fidelity testing without mock templates.

## Setup & Helpers

In [60]:
import sys
import os
import shutil
import pandas as pd
import numpy as np
from datetime import datetime

# Ensure project root is in system path
project_dir = r'D:\ex_work\AirQualityReview_Project'
if project_dir not in sys.path:
    sys.path.append(project_dir)

import analysis_logic
import audit_trail

# Setup test data workspace directory
test_workspace = os.path.join(project_dir, 'data', 'validation_tests')
os.makedirs(test_workspace, exist_ok=True)
print("Validation test workspace configured at:", test_workspace)

Validation test workspace configured at: D:\ex_work\AirQualityReview_Project\data\validation_tests


# Part 1: Module & Unit Verification (Appendix 2)

### MT-01: Software Installation & Env Verification

In [61]:
print("--- Executing MT-01: Environment Initialization & Self-Healing ---")

# 1. Verify existence of build files
exe_info_path = os.path.join(project_dir, 'app_version_info.txt')
if os.path.exists(exe_info_path):
    with open(exe_info_path, 'r') as f:
        print("App Version Info Contents:\n", f.read().strip())

# 2. Test self-healing: Clear logs and reports contents to verify they are recreated/usable
temp_reports = os.path.join(project_dir, 'reports')
temp_logs = os.path.join(project_dir, 'logs')

for path in [temp_reports, temp_logs]:
    if os.path.exists(path):
        for f in os.listdir(path):
            f_path = os.path.join(path, f)
            try:
                if os.path.isfile(f_path): os.remove(f_path)
            except Exception as e: print(f"Skipping file {f}: {e}")

print("Cleared logs and reports folder contents to test self-healing.")

# Trigger recreations
os.makedirs(temp_reports, exist_ok=True)
os.makedirs(temp_logs, exist_ok=True)
audit_trail.log_event("TEST_SELF_HEAL", "Folders successfully self-healed")

print("Is reports folder present?:", os.path.exists(temp_reports))
print("Is logs folder present?:", os.path.exists(temp_logs))
print("Is audit_trail.log created?:", os.path.exists(os.path.join(temp_logs, 'audit_trail.log')))

# .venv\Scripts\Activate.ps1
# pyinstaller AQR_Dashboard_v1.1.0_Fix.spec

--- Executing MT-01: Environment Initialization & Self-Healing ---
App Version Info Contents:
 # IQ-TC-01: Binary Integrity & Versioning Verification
#
# For more details about fixed file info:
# http://msdn.microsoft.com/en-us/library/ms646997.aspx
VSVersionInfo(
  ffi=FixedFileInfo(
    filevers=(1, 1, 0, 0),
    prodvers=(1, 1, 0, 0),
    mask=0x3f,
    flags=0x0,
    OS=0x40004,
    fileType=0x1,
    subtype=0x0,
    date=(0, 0)
    ),
  kids=[
    StringFileInfo(
      [
      StringTable(
        u'040904B0',
        [StringStruct(u'CompanyName', u'AQR Program'),
        StringStruct(u'FileDescription', u'Air Quality Review System - GAMP 5 Compliant'),
        StringStruct(u'FileVersion', u'1.1.0'),
        StringStruct(u'InternalName', u'AirQualityReview'),
        StringStruct(u'LegalCopyright', u'Ã‚Â© 2026 AQR Program. All rights reserved.'),
        StringStruct(u'OriginalFilename', u'AirQualityReview_1.1.0.exe'),
        StringStruct(u'ProductName', u'Air Quality Review'),
 

### MT-02: Cryptographic Hashing (`get_file_hash`)

In [62]:
print("--- Executing MT-02: File Hash Traceability ---")

# Setup a temporary file
test_file = os.path.join(test_workspace, 'mt02_test.txt')
with open(test_file, 'w') as f:
    f.write("Valid calibration data context.")

# Calculate hash
hash_orig = analysis_logic.get_file_hash(test_file)
print("Original File Hash:", hash_orig)
assert len(hash_orig) == 64, "Hash must be 64 characters hexadecimal"

# Modify file
with open(test_file, 'w') as f:
    f.write("Tampered calibration data context.")
hash_mod = analysis_logic.get_file_hash(test_file)
print("Modified File Hash:", hash_mod)
assert hash_orig != hash_mod, "Hash must change upon file modification"

# Test invalid file path
err_hash = analysis_logic.get_file_hash(os.path.join(test_workspace, 'non_existent.txt'))
print("Invalid File Hash Response:", err_hash)
assert err_hash.startswith("ERROR:"), "Must return an error message string"


--- Executing MT-02: File Hash Traceability ---
Original File Hash: 3852bbdd4e7173492eb07202e62c8622a2435d51f456fe7512cf58b57154f25c
Modified File Hash: f462b990918f748df0654e7d80b6b358df8803b919362213784cffd73459902a
Invalid File Hash Response: ERROR: [Errno 2] No such file or directory: 'D:\\ex_work\\AirQualityReview_Project\\data\\validation_tests\\non_existent.txt'


### MT-03: Header Row Detection (`find_header`)

In [63]:
print("--- Executing MT-03: find_header ---")

# Header at index 0
lines_1 = ["<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_1 = analysis_logic.find_header(lines_1)
print("Header at beginning row index:", idx_1)
assert idx_1 == 0

# Header after multiple rows
lines_2 = ["BAS System Log File", "Generated on Friday", "<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_2 = analysis_logic.find_header(lines_2)
print("Header after multiple rows index:", idx_2)
assert idx_2 == 2

# Missing header
lines_3 = ["Wrong header row name", "Time,Value"]
idx_3 = analysis_logic.find_header(lines_3)
print("Missing header index response:", idx_3)
assert idx_3 is None

print('---------------')
for i in lines_1:
    print(i)
print('---------------')
for i in lines_1:
    print(i)

--- Executing MT-03: find_header ---
Header at beginning row index: 0
Header after multiple rows index: 2
Missing header index response: None
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5


### MT-04: Dynamic Room Point Mapping (`find_point_mapping`)

In [64]:
print("--- Executing MT-04: find_point_mapping ---")

headers = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1P040 ROOM TEMP","","5 minutes"',
    '"Point_2:","1P040 ROOM .RMH","","5 minutes"',
    '"Point_3:","1P040 ROOM PRES","","5 minutes"',

]

headers_2 = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1-P040 ROOM PRES","","5 minutes"',
]

print('-----------------')
test_01 = analysis_logic.find_point_mapping(headers, "1-P040", "TEMP")
print("Return:", test_01)

test_02 = analysis_logic.find_point_mapping(headers, "1-P040", ".RMH")
print("Return:", test_02)

test_03 = analysis_logic.find_point_mapping(headers, "1-P040", "HUM")
print("Return:", test_03)

print('-----------------')
test_04 = analysis_logic.find_point_mapping(headers_2, "1-P999", "PRES")
print("Return:", test_04)

print('-----------------')
test_05 = analysis_logic.find_point_mapping("TEMP", "1P040", "PRES")
print("Return:", test_05)

# print('-----------------')
# test_05 = analysis_logic.find_point_mapping()
# print("Return:", test_05)

--- Executing MT-04: find_point_mapping ---
-----------------
Return: Point_1
Return: Point_2
Return: Point_2
-----------------
Return: None
-----------------
Return: None


### MT-05: Continuous Index Sequence Delineation (`find_continuous_ranges`)

In [65]:
print("--- Executing MT-05: find_continuous_ranges ---")

# Sequential values
seq = [1, 2, 3, 5, 6, 10, 11, 13]
seq_2 = [1, 3, 5, 7, 9, 11]
seq_3 = []
ranges = analysis_logic.find_continuous_ranges(seq, min_length=2)
print("Continuous ranges:", ranges)
assert ranges == [(1, 3), (5, 6), (10, 11)]

ranges_2 = analysis_logic.find_continuous_ranges(seq_2, min_length=2)
print("Un-continuous ranges:", ranges_2)

# Empty list
ranges_3 = analysis_logic.find_continuous_ranges(seq_3, min_length=2)
print("Empty ranges:", ranges_3)

--- Executing MT-05: find_continuous_ranges ---
Continuous ranges: [(1, 3), (5, 6), (10, 11)]
Un-continuous ranges: []
Empty ranges: []


### MT-06: Date Bound Extraction (`get_file_date_range`)

In [66]:
print("--- Executing MT-06: get_file_date_range ---")

# Copy standard file and check
csv_01 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
csv_02 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P041_05-14-26_01-00.csv"
csv_03 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P042_05-14-26_01-00.csv"
csv_04 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P043_05-14-26_01-00.csv"
# shutil.copyfile(src_csv, dest_csv)

start_d, end_d = analysis_logic.get_file_date_range(csv_01)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_02)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_03)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_04)
print(f"Parsed valid range: {start_d} to {end_d}")

--- Executing MT-06: get_file_date_range ---
Parsed valid range: 2026-05-29 to 2026-05-31
Parsed valid range: 2026-05-31 to 2026-05-31
Parsed valid range: 2001-01-01 to 2099-12-31
Parsed valid range: None to None


### MT-07: Standardized DataFrame Preparation (`prepare_df`)

In [67]:
print("--- Executing MT-07: prepare_df ---")

src_csv = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path)

df = analysis_logic.prepare_df(src_csv, "1-P040", setpoint_df)
print("Reindexed DataFrame Columns:", df.columns.tolist())
print("Types in DataFrame Temperature:", df['Temperature'].dtype)
print("First 5 rows:\n", df.head(10))
assert 'Temperature' in df.columns
assert 'Humidity' in df.columns
assert 'Pressure' in df.columns


--- Executing MT-07: prepare_df ---
Reindexed DataFrame Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Types in DataFrame Temperature: float64
First 5 rows:
 0            DateTime  Temperature  Humidity  Pressure
0 2026-05-29 00:00:00         21.5      41.3      12.9
1 2026-05-30 00:05:00         21.5      41.5      12.9
2 2026-05-31 00:10:00         21.5      41.5      12.9


### MT-08: Pressure Corridor Mapping (`find_compare_path`)

In [68]:
print("--- Executing MT-08: find_compare_path ---")

limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path).dropna(subset=['Room_number'])

file_list = [
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv",
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P999_01-05-26_01-00.csv"
]

# Room 1-P040 comparison corridor is 1-P051
comp_room, comp_path = analysis_logic.find_compare_path(file_list, setpoint_df, "1-P040")
print(f"Room 1-P040 Comparison Room: {comp_room}")
print(f"Comparison File Path: {comp_path}")
# assert comp_room == "1-P051"
# assert comp_path is not None


--- Executing MT-08: find_compare_path ---
Room 1-P040 Comparison Room: 1-P051
Comparison File Path: D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv


### MT-09: Legacy Date Parsing & Offset (`parse_filename_for_datetime`)

In [69]:
print("--- Executing MT-09: parse_filename_for_datetime ---")

filename = "1-P040_05-30-26_00-00.csv"
parsed_date = analysis_logic.parse_filename_for_datetime(filename)
print("Filename: ", filename)
print("Parsed previous-day business date: ", parsed_date)
# 05-30-26 parsed is 2026-05-30. Previous day rule makes it 2026-05-29
assert parsed_date == datetime(2026, 5, 29).date()


--- Executing MT-09: parse_filename_for_datetime ---
Filename:  1-P040_05-30-26_00-00.csv
Parsed previous-day business date:  2026-05-29


### MT-10: Phase II Unified Data Generation (`prepare_df_phase2`)

In [70]:
print("--- Executing MT-10: prepare_df_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b11\2026-05-01"
limit_path_p2 = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit_B11.xlsx"
setpoint_df_p2 = pd.read_excel(limit_path_p2)

room_id, df_p2, sensors = analysis_logic.prepare_df_phase2(phase2_room_dir, "11-1-012", setpoint_df_p2)
print("Unified Room ID:", room_id)
print("Merged Columns:", df_p2.columns.tolist())
print("Discovered Sensors:", sensors)
display(df_p2.head())
assert 'Temperature' in df_p2.columns
assert 'Humidity' in df_p2.columns
assert 'Pressure' in df_p2.columns


--- Executing MT-10: prepare_df_phase2 ---


Unified Room ID: 11-1-012
Merged Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Discovered Sensors: {'Temperature', 'Pressure', 'Humidity'}


,DateTime,Temperature,Humidity,Pressure
0,2026-04-30 08:00:00,22.47,49.04,15.46
1,2026-04-30 08:05:00,22.44,48.53,14.12
2,2026-04-30 08:10:00,22.44,48.53,16.90
3,2026-04-30 08:15:00,22.40,48.79,15.92
4,2026-04-30 08:20:00,22.40,48.79,15.94


### MT-11: Phase II Multiple Files Date Extraction (`get_file_date_range_phase2`)

In [71]:
print("--- Executing MT-11: get_file_date_range_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b12"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-073")
print(f"Phase II date range: {s_date} to {e_date}")

s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-074")
print(f"Phase II date range: {s_date} to {e_date}")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b12_blank"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-074")
print(f"Phase II date range: {s_date} to {e_date}")

--- Executing MT-11: get_file_date_range_phase2 ---
Phase II date range: 2026-05-29 to 2026-05-31
Phase II date range: 2026-05-29 to 2026-05-31
Phase II date range: None to None


# Part 2: Integration, System Transformation & Error Verification (Appendix 3)

### ERR-001: Missing Header Row Detection

In [72]:
print("--- Executing ERR-001: Missing header check ---")

err_dir = os.path.join(test_workspace, 'case_err001')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file missing <>Date
with open(err_file, 'w') as f:
    f.write('BAS Log file\nWrongHeaderRow,Time,Point_1\n05/30/2026,00:00,22.5\n')

try:
    analysis_logic.prepare_df(err_file, "1-P045")
    print("FAIL: Expected error was not raised")
except ValueError as e:
    print("PASS: Expected error message caught:", e)
    assert "ERR-001" in str(e)


--- Executing ERR-001: Missing header check ---
PASS: Expected error message caught: ERR-001: Critical Error - Header '<>Date' not found in file: D:\ex_work\AirQualityReview_Project\data\validation_tests\case_err001\1-P040_05-30-26_00-00.csv


### ERR-002: Missing Limit File Detection

In [73]:
print("--- Executing ERR-002: Missing limit file ---")

folder_path = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C"
missing_limit_path = r"D:\ex_work\AirQualityReview_Project\data\MissingLimit.xlsx"

out, logs, plot = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=missing_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Analysis return output path:", out)
print("Log output:\n", logs)
assert out is None
assert "ERR-002" in logs


--- Executing ERR-002: Missing limit file ---
Analysis return output path: None
Log output:
 ERROR: ERR-002: Limit File Not Found
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 792, in analyze_files
    setpoint_df = pd.read_excel(setpoint_path)
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 481, in read_excel
    io = ExcelFile(
        io,
    ...<2 lines>...
        engine_kwargs=engine_kwargs,
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1604, in __init__
    ext = inspect_excel_format(
        content_or_path=path_or_buffer, storage_options=storage_options
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1452, in inspect_excel_format
    with get_handle(
         ~~~~~~~~~~^
        content_or_path, "rb", storage_options=storage_options, is_text=False
     

### ERR-003: Invalid Configuration DataType (Non-Numeric)

In [74]:
print("--- Executing ERR-003: Non-numeric limit checks ---")

err_limit_dir = os.path.join(test_workspace, 'case_err003')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err003.xlsx')

# Load standard limit and inject non-numeric
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit['Temperature_Limit'] = df_limit['Temperature_Limit'].astype(object)
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Temperature_Limit'] = "NonNumericLimit"
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
# assert "ERR-003" in logs, "Logs should capture the non-numeric limit GxP warning"


--- Executing ERR-003: Non-numeric limit checks ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-004: Audit Trail Tampering Detection

In [75]:
print("--- Executing ERR-004: Audit log tampering ---")

log_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log"
backup_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log.tmp_bak"

# 1. Backup log
if os.path.exists(log_file):
    shutil.copyfile(log_file, backup_file)
    
    # 2. Tamper log
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write('{"timestamp": "2026-06-04 10:00:00", "user": "attacker", "action": "TAMPER", "prev_hash": "wrong", "entry_hash": "invalid"}\n')

    try:
        valid, msg = audit_trail.verify_audit_trail()
        print(f"Verification status: {valid} | Msg: {msg}")
        assert not valid, "Validation must detect tampering"
        print("PASS: Tampering successfully detected.")
    finally:
        # 3. Restore backup
        shutil.copyfile(backup_file, log_file)
        os.remove(backup_file)
        print("Restored audit log backup.")


--- Executing ERR-004: Audit log tampering ---
Verification status: False | Msg: Broken chain at line 5: Hash mismatch.
PASS: Tampering successfully detected.
Restored audit log backup.


### ERR-005: Missing Parameter Raw Data or Columns

In [76]:
print("--- Executing ERR-005: Upfront missing raw files & columns validation ---")

# Test Phase I Missing raw file
out1, logs1, plot1 = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P032"],  # Exists in limit but no CSV in folder
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Phase I Missing file output logs:\n", logs1)
assert out1 is None
assert "ERR-005" in logs1

# Test Phase II Missing sensor files
dummy_scan_dir = os.path.join(test_workspace, 'dummy_p2')
os.makedirs(os.path.join(dummy_scan_dir, '1-P040'), exist_ok=True)
# Only humidity file exists, temp (RMT) is missing
with open(os.path.join(dummy_scan_dir, '1-P040', '1-P040_RMH_2026-05-30_Mock.csv'), 'w') as f:
    f.write("DateTime;Data Source;Value\n2026-05-30 00:00:00;Room Hum;45.2\n")

out2, logs2, plot2 = analysis_logic.analyze_files_phase2(
    folder_path=dummy_scan_dir,
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(dummy_scan_dir)
print("Phase II Missing sensor file logs:\n", logs2)
assert out2 is None
assert "ERR-005" in logs2


--- Executing ERR-005: Upfront missing raw files & columns validation ---
Phase I Missing file output logs:
 ERROR: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 861, in analyze_files
    raise ValueError(f"ERR-005: Raw data file not found in {folder_path} for Room {room_id}")
ValueError: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032

Phase II Missing sensor file logs:
 FILE ERROR [1-P040]: ERR-005: No Temperature file (_RMT_) found in D:\ex_work\AirQualityReview_Project\data\validation_tests\dummy_p2 for 1-P040
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 1335, in analyze_files_phase2
    _, df, sensors = prepare_df_phase2(raw_data_path, room_id=room_id, setpoint_df=setpoint_df)
                     ~~~~~~~~~~~

### ERR-006: Inverted Limits Configuration (High < Low)

In [77]:
print("--- Executing ERR-006: Logical constraint check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err006')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err006.xlsx')

# Create limit file with Low Limit > High Limit
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_High_Limit'] = 20
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_Low_Limit'] = 50
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
# assert "ERR-006" in logs, "Logs should capture GxP configuration limit inversion warning"


--- Executing ERR-006: Logical constraint check ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-007: Excel Report Generation Failure

In [78]:
print("--- Executing ERR-007: Simulate Report Write Lock ---")

# We mock standard ExcelWriter to raise PermissionError and verify ERR-007
import unittest.mock as mock

with mock.patch('pandas.ExcelWriter', side_effect=Exception("Write Lock Permission Denied")):
    out, logs, plot = analysis_logic.analyze_files(
        folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
        setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
        selected_rooms=["1-P040"],
        start_date="2026-05-13",
        end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is None
# assert "ERR-007" in logs


--- Executing ERR-007: Simulate Report Write Lock ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-008: Identical Duplicate Timestamps

In [79]:
print("--- Executing ERR-008: Duplicate Timestamp Deduplication ---")

err_dir = os.path.join(test_workspace, 'case_err008')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file with identical duplicate timestamps
csv_dup = (
    '"Key            Name:Suffix                                Trend Definitions Used"\n'
    '"Point_1:","1-P040 ROOM TEMP","","5 minutes"\n'
    '"Point_2:","1-P040 ROOM HUM","","5 minutes"\n'
    '"Point_3:","1-P040 ROOM PRES","","5 minutes"\n'
    '"<>Date","Time","Point_1","Point_2","Point_3"\n'
    '"05/30/2026","00:00","22.5","45.0","40.0"\n'
    '"05/30/2026","00:00","23.0","45.0","40.0"\n'
)
with open(err_file, 'w') as f:
    f.write(csv_dup)

df = analysis_logic.prepare_df(err_file, "1-P040")
print("Deduplicated DataFrame size:", len(df))
print("DataFrame contents:\n", df)
assert len(df) == 1, "Must drop duplicate timestamps and keep only the first record"


--- Executing ERR-008: Duplicate Timestamp Deduplication ---
[WARNING] ERR-008: Duplicate Timestamps Detected in file 1-P040_05-30-26_00-00.csv (Room 1-P040). Found 1 duplicate timestamps from 2026-05-30 00:00:00 to 2026-05-30 00:00:00. Dropping duplicates and keeping the first record.
Deduplicated DataFrame size: 1
DataFrame contents:
 0   DateTime  Temperature  Humidity  Pressure
0 2026-05-30         22.5      45.0      40.0


### ERR-009: Invalid Limit File Structure (Missing Columns)

In [80]:
print("--- Executing ERR-009: Missing columns in limits check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err009')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err009.xlsx')

# Drop a required column
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit = df_limit.drop(columns=['Temperature_Limit'])
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is None
assert "ERR-009" in logs


--- Executing ERR-009: Missing columns in limits check ---
Log output:
 ERROR: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 799, in analyze_files
    raise ValueError(f"ERR-009: Invalid Limit File Format - Missing required columns: {', '.join(missing_cols)}")
ValueError: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit



### ERR-010: Cross-Uploading Data Screens

In [81]:
print("--- Executing ERR-010: Cross-upload checks ---")

# Set up a temp folder with only Phase II files in it
err010_dir = os.path.join(test_workspace, 'case_err010')
os.makedirs(os.path.join(err010_dir, '1-P040'), exist_ok=True)
with open(os.path.join(err010_dir, '1-P040', '1-P040_RMT_2026-05-30.csv'), 'w') as f:
    f.write("DateTime;Value\n2026-05-30 00:00:00;22.5\n")

# Try to feed Phase II directories folder to Phase I analyze_files
out, logs, plot = analysis_logic.analyze_files(
    folder_path=err010_dir,  # Contains Phase II structure
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(err010_dir)
print("Log output:\n", logs)
assert out is None
assert "ERR-010" in logs or "ERR-005" in logs or "No Matching Files" in logs, "Must trigger validation/matching failure"


--- Executing ERR-010: Cross-upload checks ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-000: Data Loss Remark

In [82]:
print("--- Executing ERR-000: Data Loss Flagging ---")

df_data = pd.DataFrame({
    'DateTime': pd.date_range("2026-05-30 00:00:00", periods=6, freq='5min'),
    'Temperature': [22.0, 22.1, np.nan, 22.0, 22.3, 22.1], # NaN value introduced
    'Humidity': [45.0, 45.2, 45.1, 45.0, 45.3, 45.1],
    'Pressure': [40.0, 40.2, 40.1, 40.0, 40.3, 40.1]
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [35.0],
    'Pressure_High_Limit': [45.0]
})

spec_txt, res_txt = analysis_logic._analyze_single_room_core(
    df_data, "1-P040", setpoint_row, "Passed", "Out of spec",
    set(), {}, ['1-P040'], setpoint_row,
    pd.Timestamp("2026-05-30 00:00:00"), pd.Timestamp("2026-05-30 00:25:00")
)
print("Analysis Results text:\n", res_txt)
assert "Data Loss" in res_txt, "Must append Data Loss warning tag"


--- Executing ERR-000: Data Loss Flagging ---
Temperature Data Loss Found for 1-P040:
           DateTime Temperature
2026-05-30 00:10:00   Data Loss
-------------------------------
Analysis Results text:
 Temperature: Data Loss
00:10 to 00:10
Humidity: Passed
Pressure: Passed


# Integration / System Logic Verification

### InT-01: check_reverse_violations (REAL data + copied validation data)


In [ ]:
print("=" * 80)
print("InT-01: check_reverse_violations - real-data based execution")
print("=" * 80)

from pathlib import Path
import io
import os
import shutil
from contextlib import redirect_stdout

import pandas as pd
import openpyxl

import analysis_logic

PROJECT_ROOT = Path(r"D:\ex_work\AirQualityReview_Project")
DATA_ROOT = PROJECT_ROOT / "data"
BAS_SOURCE = DATA_ROOT / "csv_main" / "C"
SETPOINT_PATH = DATA_ROOT / "SetPointLimit.xlsx"
VALIDATION_ROOT = DATA_ROOT / "validation_tests" / "int_01_06_realdata"
PHASE2_SOURCE = DATA_ROOT / "csv_b11"
PHASE2_SETPOINT_PATH = DATA_ROOT / "SetPointLimit_B11.xlsx"


def reset_case_dir(name):
    case_dir = VALIDATION_ROOT / name
    if case_dir.exists():
        shutil.rmtree(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def copy_bas_file(room_id, dest_dir):
    matches = sorted(BAS_SOURCE.glob(f"{room_id}_*.csv"))
    assert matches, f"No BAS source CSV found for {room_id}"
    dest = dest_dir / matches[0].name
    shutil.copy2(matches[0], dest)
    return dest


def load_setpoint_copy(dest_dir, filename="SetPointLimit_case.xlsx"):
    dest = dest_dir / filename
    shutil.copy2(SETPOINT_PATH, dest)
    return pd.read_excel(dest).dropna(subset=["Room_number"]), dest


def save_setpoint(df, path):
    df.to_excel(path, index=False)


def prepared_from_copy(csv_path, room_id, setpoint_df):
    return analysis_logic.prepare_df(str(csv_path), room_id, setpoint_df)


def assert_contains(text, needle, case_id):
    assert needle in text, f"{case_id} expected {needle!r} in {text!r}"


def excel_contains_text(path, needle):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    try:
        for ws in wb.worksheets:
            for row in ws.iter_rows(values_only=True):
                for value in row:
                    if value is not None and needle in str(value):
                        return True
        return False
    finally:
        wb.close()


def safe_text(value, limit=None):
    text = str(value)
    if limit is not None:
        text = text[:limit]
    return text.encode("ascii", "ignore").decode("ascii")


case_dir = reset_case_dir("int01")
setpoint_df, setpoint_case_path = load_setpoint_copy(case_dir)
room_csv = copy_bas_file("1-P045", case_dir)
corridor_csv = copy_bas_file("1-P051", case_dir)
file_list = [str(room_csv), str(corridor_csv)]

room_df = prepared_from_copy(room_csv, "1-P045", setpoint_df)
corridor_df = prepared_from_copy(corridor_csv, "1-P051", setpoint_df)
base_times = corridor_df["DateTime"].head(3).tolist()

print("--- InT-01-1: high-pressure dependent room drops below corridor -> OVER ---")
sp_high = setpoint_df.copy()
sp_high.loc[sp_high["Room_number"].astype(str) == "1-P045", ["Pressure_Low_Limit", "Pressure_High_Limit"]] = [35, 45]
high_room_df = room_df.copy()
high_room_df.loc[:2, "DateTime"] = base_times
high_room_df.loc[:2, "Pressure"] = [20, 20, 20]
high_corridor_df = corridor_df.copy()
high_corridor_df.loc[:2, "Pressure"] = [30, 31, 32]
buf = io.StringIO()
with redirect_stdout(buf):
    lines_01_1 = analysis_logic.check_reverse_violations(
        "1-P051", high_corridor_df, 0, 2, sp_high, ["1-P045", "1-P051"], {"1-P045": high_room_df, "1-P051": high_corridor_df}
    )
log_01_1 = buf.getvalue()
print(log_01_1)
print(lines_01_1)
assert lines_01_1 and "over 1-P045" in "".join(lines_01_1)
assert "REVERSE OVER violation data" in log_01_1
print("InT-01-1 PASS")

print("\n--- InT-01-2: low-pressure dependent room spikes above corridor -> UNDER ---")
low_room_df = room_df.copy()
low_room_df.loc[:2, "DateTime"] = base_times
low_room_df.loc[:2, "Pressure"] = [40, 41, 42]
low_corridor_df = corridor_df.copy()
low_corridor_df.loc[:2, "Pressure"] = [30, 31, 32]
buf = io.StringIO()
with redirect_stdout(buf):
    lines_01_2 = analysis_logic.check_reverse_violations(
        "1-P051", low_corridor_df, 0, 2, setpoint_df, ["1-P045", "1-P051"], {"1-P045": low_room_df, "1-P051": low_corridor_df}
    )
log_01_2 = buf.getvalue()
print(log_01_2)
print(lines_01_2)
assert lines_01_2 and "under 1-P045" in "".join(lines_01_2)
assert "REVERSE UNDER violation data" in log_01_2
print("InT-01-2 PASS")

print("\n--- InT-01-3: merge_asof tolerance keeps <=60s and drops >60s ---")
tolerance_room_df = room_df.head(2).copy()
tolerance_corridor_df = corridor_df.head(2).copy()
tolerance_corridor_df.loc[0, "DateTime"] = pd.Timestamp("2026-05-13 00:00:00")
tolerance_corridor_df.loc[1, "DateTime"] = pd.Timestamp("2026-05-13 00:05:00")
tolerance_room_df.loc[0, "DateTime"] = pd.Timestamp("2026-05-13 00:00:45")
tolerance_room_df.loc[1, "DateTime"] = pd.Timestamp("2026-05-13 00:06:30")
tolerance_corridor_df.loc[:, "Pressure"] = [30, 30]
tolerance_room_df.loc[:, "Pressure"] = [40, 40]
buf = io.StringIO()
with redirect_stdout(buf):
    lines_01_3 = analysis_logic.check_reverse_violations(
        "1-P051", tolerance_corridor_df, 0, 1, setpoint_df, ["1-P045", "1-P051"], {"1-P045": tolerance_room_df, "1-P051": tolerance_corridor_df}
    )
log_01_3 = buf.getvalue()
print(log_01_3)
print(lines_01_3)
assert "00:00" in "".join(lines_01_3)
assert "00:05" not in "".join(lines_01_3)
print("InT-01-3 PASS")

print("\n--- InT-01-4: find_compare_path extracts room id and returns configured corridor tuple ---")
compare_room, compare_path = analysis_logic.find_compare_path(file_list, setpoint_df, "1-P045")
print(compare_room, compare_path)
assert compare_room == "1-P051"
assert compare_path and Path(compare_path).name.startswith("1-P051")
print("InT-01-4 PASS")

print("\nInT-01 COMPLETE")


### InT-02: _compute_plot_result (REAL prepare_df data + copied value adjustments)


In [ ]:
print("=" * 80)
print("InT-02: _compute_plot_result - real prepare_df data with controlled copied values")
print("=" * 80)

case_dir = reset_case_dir("int02")
setpoint_df, _ = load_setpoint_copy(case_dir)
room_csv = copy_bas_file("1-P045", case_dir)
df_real = prepared_from_copy(room_csv, "1-P045", setpoint_df)
start_dt = df_real["DateTime"].min()
end_dt = start_dt + pd.Timedelta(minutes=35)

print("--- InT-02-1: 6 continuous rows create one temp and one humidity violation; pressure no spec -> 0 ---")
sp_no_pressure = setpoint_df.copy()
mask = sp_no_pressure["Room_number"].astype(str) == "1-P045"
sp_no_pressure.loc[mask, ["Pressure_Low_Limit", "Pressure_High_Limit"]] = [pd.NA, pd.NA]
df_case = df_real.head(8).copy()
df_case["Temperature"] = 24.0
df_case["Humidity"] = 45.0
df_case.loc[:5, "Temperature"] = 26.0
df_case.loc[:5, "Humidity"] = 60.0
res = analysis_logic._compute_plot_result({"1-P045": [df_case]}, sp_no_pressure, ["1-P045"], start_dt, end_dt)
print(res["summary"])
s = res["summary"][0]
assert s["temp_v"] == 1 and s["hum_v"] == 1 and s["press_v"] == 0, res["summary"]
print("InT-02-1 PASS")

print("\n--- InT-02-2: fewer than 6 continuous rows are filtered out ---")
df_short = df_real.head(8).copy()
df_short["Temperature"] = 24.0
df_short["Humidity"] = 45.0
df_short["Pressure"] = 15.0
df_short.loc[:4, "Temperature"] = 26.0
df_short.loc[:4, "Humidity"] = 60.0
df_short.loc[:4, "Pressure"] = 40.0
res_short = analysis_logic._compute_plot_result({"1-P045": [df_short]}, setpoint_df, ["1-P045"], start_dt, end_dt)
print(res_short["summary"])
s = res_short["summary"][0]
assert s["temp_v"] == 0 and s["hum_v"] == 0 and s["press_v"] == 0, res_short["summary"]
print("InT-02-2 PASS")

print("\n--- InT-02-3: missing values do not crash and are returned as None in plot_data ---")
df_missing = df_real.head(8).copy()
df_missing.loc[1, "Temperature"] = pd.NA
df_missing.loc[2, "Humidity"] = pd.NA
df_missing.loc[3, "Pressure"] = pd.NA
res_missing = analysis_logic._compute_plot_result({"1-P045": [df_missing]}, setpoint_df, ["1-P045"], start_dt, end_dt)
plot_data = res_missing["plot_data"]["1-P045"]
print(pd.DataFrame({"time": plot_data["times"][:5], "temp": plot_data["temp"][:5], "hum": plot_data["hum"][:5], "press": plot_data["press"][:5]}))
assert plot_data["temp"][1] is None
assert plot_data["hum"][2] is None
assert plot_data["press"][3] is None
print("InT-02-3 PASS")

print("\nInT-02 COMPLETE")


### InT-03: get_plot_info (REAL directory scan + corrupt copied CSV)


In [ ]:
print("=" * 80)
print("InT-03: get_plot_info - real directory scan and real prepare_df error handling")
print("=" * 80)

case_dir = reset_case_dir("int03")
nested_dir = case_dir / "nested" / "bas"
nested_dir.mkdir(parents=True, exist_ok=True)
setpoint_df, setpoint_case_path = load_setpoint_copy(case_dir)
for room in ["1-P045", "1-P051"]:
    copy_bas_file(room, nested_dir)

print("--- InT-03-1: nested scan, room parsing, selected-room filtering ---")
res = analysis_logic.get_plot_info(
    folder_path=str(case_dir),
    setpoint_path=str(setpoint_case_path),
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-13",
    limits=None,
)
print(res.keys())
print(res.get("summary"))
assert "1-P045" in res.get("plot_data", {})
assert "1-P051" not in res.get("plot_data", {})
assert res.get("summary") and res["summary"][0]["room_id"] == "1-P045"
print("InT-03-1 PASS")

print("\n--- InT-03-2: corrupt real-named CSV is logged and loop continues ---")
corrupt_path = nested_dir / "1-P045_05-15-26_01-00.csv"
corrupt_path.write_text("this,is,not,a,valid,AQR,csv\n1,2,3\n", encoding="utf-8")
before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
res_corrupt = analysis_logic.get_plot_info(
    folder_path=str(case_dir),
    setpoint_path=str(setpoint_case_path),
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-13",
    limits=None,
)
after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
new_audit = after[len(before):]
print(new_audit[-2000:])
assert "PLOT_DATA_ERROR" in new_audit
assert "1-P045_05-15-26_01-00.csv" in new_audit
assert "1-P045" in res_corrupt.get("plot_data", {})
print("InT-03-2 PASS")

print("\nInT-03 COMPLETE")


### InT-04: _analyze_single_room_core (REAL prepare_df data + controlled copied values)


In [ ]:
print("=" * 80)
print("InT-04: _analyze_single_room_core - real prepare_df data with copied value adjustments")
print("=" * 80)

case_dir = reset_case_dir("int04")
setpoint_df, _ = load_setpoint_copy(case_dir)
room_csv = copy_bas_file("1-P045", case_dir)
corridor_csv = copy_bas_file("1-P051", case_dir)
df_room_base = prepared_from_copy(room_csv, "1-P045", setpoint_df).head(12).copy()
df_corr_base = prepared_from_copy(corridor_csv, "1-P051", setpoint_df).head(12).copy()
row_045 = setpoint_df[setpoint_df["Room_number"].astype(str) == "1-P045"]
tick_mark = "Pass"
cross_mark = "Fail"
day_start = df_room_base["DateTime"].min()
day_end = df_room_base["DateTime"].max()


def run_core(df, prepared_cache):
    captured = io.StringIO()
    with redirect_stdout(captured):
        spec, result = analysis_logic._analyze_single_room_core(
            df, "1-P045", row_045, tick_mark, cross_mark, ["1-P051"],
            prepared_cache, ["1-P045", "1-P051"], setpoint_df, day_start, day_end
        )
    print(captured.getvalue().encode("ascii", "ignore").decode("ascii")[:3000])
    return spec, result

print("--- InT-04-1: >=25-minute block fails; transient block is omitted ---")
df_25 = df_room_base.copy()
df_25["Temperature"] = 24.0
df_25.loc[0:5, "Temperature"] = 26.0
df_25.loc[8:10, "Temperature"] = 26.0
spec, result = run_core(df_25, {"1-P045": df_25, "1-P051": df_corr_base})
print(result)
assert "Temperature: Fail" in result
assert "00:00 to 00:25" in result
assert "00:40 to 00:50" not in result
print("InT-04-1 PASS")

print("\n--- InT-04-2: NaN sensor readings are flagged as Data Loss and concatenate with Fail ---")
df_loss = df_25.copy()
df_loss.loc[7, "Temperature"] = pd.NA
df_loss.loc[8, "Humidity"] = pd.NA
spec, result_loss = run_core(df_loss, {"1-P045": df_loss, "1-P051": df_corr_base})
print(result_loss)
assert "Temperature: Fail & Data Loss" in result_loss
assert "Humidity:" in result_loss and "Data Loss" in result_loss
print("InT-04-2 PASS")

print("\n--- InT-04-3: pressure out-of-limit can downgrade to false alarm or trigger fail via corridor comparison ---")
df_false_alarm = df_room_base.copy()
df_corr = df_corr_base.copy()
df_false_alarm["Pressure"] = 40.0
df_corr["Pressure"] = 45.0
spec, result_false = run_core(df_false_alarm, {"1-P045": df_false_alarm, "1-P051": df_corr})
print(result_false)
assert "Pressure: Pass" in result_false and "within 1-P051" in result_false

df_actual_fail = df_room_base.copy()
df_actual_fail["Pressure"] = 40.0
df_corr["Pressure"] = 30.0
spec, result_fail = run_core(df_actual_fail, {"1-P045": df_actual_fail, "1-P051": df_corr})
print(result_fail)
assert "Pressure: Fail" in result_fail and "over 1-P051" in result_fail
print("InT-04-3 PASS")

print("\nInT-04 COMPLETE")


### InT-05: analyze_files (Phase I REAL full process with validation copies)


In [ ]:
print("=" * 80)
print("InT-05: analyze_files - Phase I full statistical execution with real-data copies")
print("=" * 80)

case_dir = reset_case_dir("int05")
setpoint_df, setpoint_case_path = load_setpoint_copy(case_dir)
for room in ["1-P045", "1-P051"]:
    copy_bas_file(room, case_dir)

before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=str(case_dir),
    setpoint_path=str(setpoint_case_path),
    selected_rooms=["1-P045", "1-P051"],
    start_date="2026-05-13",
    end_date="2026-05-14",
)
after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
new_audit = after[len(before):]
print(safe_text(logs, 4000))
print("--- audit tail ---")
print(safe_text(new_audit[-4000:]))
print("--- plot_result summary ---")
print(plot_result.get("summary") if isinstance(plot_result, dict) else plot_result)
print("--- report ---")
print(out_path)

assert "ANALYSIS_START" in new_audit
assert "FILE_PROCESSED" in new_audit
assert "ANALYSIS_SUCCESS" in new_audit
assert out_path and Path(out_path).exists() and Path(out_path).stat().st_size > 0
assert isinstance(plot_result, dict) and plot_result.get("summary")
assert excel_contains_text(out_path, "Software Version: v1.1.0")
assert excel_contains_text(out_path, "1-P045")
print("InT-05-1 PASS")

print("\nInT-05 COMPLETE")


### InT-06: analyze_files_phase2 (Phase II REAL full process)


In [ ]:
print("=" * 80)
print("InT-06: analyze_files_phase2 - Phase II full statistical execution with real data")
print("=" * 80)

case_dir = reset_case_dir("int06")
phase2_case = case_dir / "csv_b11"
shutil.copytree(PHASE2_SOURCE / "2026-05-01", phase2_case / "2026-05-01")
shutil.copy2(PHASE2_SETPOINT_PATH, case_dir / "SetPointLimit_B11.xlsx")

before = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
out_path, logs, plot_result = analysis_logic.analyze_files_phase2(
    folder_path=str(phase2_case),
    setpoint_path=str(case_dir / "SetPointLimit_B11.xlsx"),
    selected_rooms=["11-1-012", "11-1-013"],
    start_date="2026-04-30",
    end_date="2026-05-01",
)
after = Path(analysis_logic.audit_trail.LOG_FILE).read_text(encoding="utf-8", errors="ignore") if Path(analysis_logic.audit_trail.LOG_FILE).exists() else ""
new_audit = after[len(before):]
print(safe_text(logs, 4000))
print("--- audit tail ---")
print(safe_text(new_audit[-4000:]))
print("--- plot_result summary ---")
print(plot_result.get("summary") if isinstance(plot_result, dict) else plot_result)

room_id, df_p2, sensors = analysis_logic.prepare_df_phase2(str(phase2_case / "2026-05-01"), "11-1-012", pd.read_excel(case_dir / "SetPointLimit_B11.xlsx"))
print("prepare_df_phase2:", room_id, df_p2.shape, sensors)

assert "ANALYSIS_START" in new_audit
assert "FILE_PROCESSED" in new_audit
assert "ANALYSIS_SUCCESS" in new_audit
assert out_path and Path(out_path).exists() and Path(out_path).stat().st_size > 0
assert isinstance(plot_result, dict) and plot_result.get("summary")
assert room_id == "11-1-012" and not df_p2.empty
assert {"Temperature", "Humidity", "Pressure"}.issubset(set(sensors))
assert excel_contains_text(out_path, "Software Version: v1.1.0")
print("InT-06-1 PASS")

print("\nInT-06 COMPLETE")
